## Document

LangChain 의 기본 문서 객체입니다.

**속성**
- `page_content`: 문서의 내용을 나타내는 문열입니다.
- `metadata`: 문서의 메타데이터를 나타내는 딕셔너리입니다.


In [ ]:
from langchain_core.documents import Document

In [8]:
document = Document(page_content="안녕하세요? 이건 랭체인의 도큐먼트입니다.")

# 도큐먼트의 속성 확인
document.__dict__

{'id': None,
 'metadata': {},
 'page_content': '안녕하세요? 이건 랭체인의 도큐먼트입니다.',
 'type': 'Document'}

In [9]:
# 메타데이터 추가 및 속성 추가
document.metadata["source"] = "LangChain Guide"
document.metadata["page"] = 1
document.metadata["author"] = "Heeyeong"

# 도큐먼트 속성 확인
document.metadata

{'source': 'LangChain Guide', 'page': 1, 'author': 'Heeyeong'}

## Document Loader

다양한 파일의 형식으로부터 불러온 내용을 문서(Document) 객체로 변환하는 역할을 합니다.

### 주요 Loader 
- PyPDFLoader: PDF 파일을 로드하는 로더입니다.
- CSVLoader: CSV 파일을 로드하는 로더입니다.
- UnstructuredHTMLLoader: HTML 파일을 로드하는 로더입니다.
- JSONLoader: JSON 파일을 로드하는 로더입니다.
- TextLoader: 텍스트 파일을 로드하는 로더입니다.
- DirectoryLoader: 디렉토리를 로드하는 로더입니다.

In [19]:
FILE_PATH = "./data/python_레퍼런스.pdf"

In [15]:
# !pip install pypdf

In [20]:
from langchain_community.document_loaders import PyPDFLoader

#로더 설정
loader = PyPDFLoader(FILE_PATH)

## Load 메소드 종류
- load()
- load and split()
- lazy_load()
- aload()


generator 방식으로 문서를 로드합니다.
: 문서를 한 번에 메모리에 전부 읽어들이지 않고, 필요할 때마다 하나씩 생성(로드)하여 처리하는 방식을 의미합니다.

- 일반 방식 (load()): 모든 페이지를 한 번에 불러와 리스트에 담습니다. 문서가 아주 크면 메모리를 많이 차지하게 됩니다.

- 분할 방식 (load_and_split()): 문서 로더(Document Loader)가 파일이나 데이터를 읽어옴과 동시에, 지정한 텍스트 분할기(Text Splitter)를 사용해 문서를 작은 조각(Chunk)으로 쪼개어 반환해 주는 기능입니다.

    - AG(검색 증강 생성) 시스템 구축 시
    
    LLM(대형 언어 모델)에 긴 문서 전체를 넣으면 토큰 제한을 초과하거나 처리 비용이 커집니다. 문서를 의미 단위나 적절한 길이로 잘라 벡터 저장소(VectorStore)에 저장해야 할 때 유용합니다.

- 제너레이터 방식 (lazy_load()): 문서를 한 번에 메모리에 전부 읽어들이지 않고, 필요할 때마다 하나씩 생성(로드)하여 처리하는 방식을 의미합니다. 파이썬의 yield 키워드처럼 문서를 순회(for 문 등)할 때마다 한 장씩 차례대로 가져옵니다. 따라서 메모리 사용량을 크게 줄일 수 있습니다.




### load()

- 문서를 로드하여 반환합니다.
- 반환된 결과는 `List[Document]` 형태입니다.

In [25]:
# PDF 로더
docs = loader.load()

# 로드된 문서의 수 확인
len(docs)

3

In [23]:
# 첫번째 문서 확인
docs[0]

Document(metadata={'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-09-17T06:14:24+00:00', 'title': '레퍼런스', 'moddate': '2026-09-17T06:14:24+00:00', 'source': './data/python_레퍼런스.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='📕\n레퍼런스\n외우지  않고  찾아보는  자리입니다 . 표 · 목록 · 문법  형식은  전부  여기로 .\n이해해야  하는  것은  개념 페이지 , 필요할  때  검색하는  것은  것은  레퍼런스\n🔗https://app.notion.com/p/3d747ecb9b23803abbb5e8abf2d722d6?\nsource=copy_link\n자주  쓰는  파이썬  함수\n텍스트  유형 : str[ 스트링 ]\n숫자  유형 : int[ 인티 ] , float , complex\n시퀀스  유형 : list[ 리스트 ] , tuple[ 튜플 ] , range[ 레인\n지 ]\n매핑  유형 : dict[ 딕트 ]\n세트  유형 : set ,frozenset\n부울  유형 : bool[ 부울 ] true/false ( 참 / 거짓 )\n이진  유형 : bytes , bytearray , memoryview\n없음  유형 : NoneType\nExample Data Type 중요도\nx = "Hello World" str O\nx = 20 int O\nx = 20.5 float O\nx = 1j complex\nx = ["apple", "banana", "cherry"] list O\nx = ("apple", "ban

### load_and_split()

- splitter 를 사용하여 문서를 분할하고 반환합니다.
- 반환된 결과는 `List[Document]` 형태입니다.

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문열 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)

# 파일 경로
FILE_PATH = "./data/python_레퍼런스.pdf"

# 로더 설정
loader = PyPDFLoader(FILE_PATH)

# 문서 분할
split_docs = loader.load_and_split(text_splitter=text_splitter)

# 로드된 문서의 수 확인
print(f"문서의 길이: {len(split_docs)}")

# 첫번째 문서 확인
split_docs[0]

문서의 길이: 9


Document(metadata={'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-09-17T06:14:24+00:00', 'title': '레퍼런스', 'moddate': '2026-09-17T06:14:24+00:00', 'source': './data/python_레퍼런스.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='📕\n레퍼런스\n외우지  않고  찾아보는  자리입니다 . 표 · 목록 · 문법  형식은  전부  여기로 .\n이해해야  하는  것은  개념 페이지 , 필요할  때  검색하는  것은  것은  레퍼런스\n🔗https://app.notion.com/p/3d747ecb9b23803abbb5e8abf2d722d6?\nsource=copy_link\n자주  쓰는  파이썬  함수')

### lazy_load()

In [31]:
loader.lazy_load()

<generator object PyPDFLoader.lazy_load at 0x00000149C8DDCE40>

In [32]:
# generator 방식으로 문서 로드
for doc in loader.lazy_load():
    print(doc.metadata)

{'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-09-17T06:14:24+00:00', 'title': '레퍼런스', 'moddate': '2026-09-17T06:14:24+00:00', 'source': './data/python_레퍼런스.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}
{'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-09-17T06:14:24+00:00', 'title': '레퍼런스', 'moddate': '2026-09-17T06:14:24+00:00', 'source': './data/python_레퍼런스.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2'}
{'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-09-17T06:14:24+00:00', 'title': '레퍼런스', 'moddate': '2026-09-17T06:14:24+00:00', 'source': './data/python_레퍼런스.pdf', 'total_pages': 3, 'page': 2,

In [39]:
# 문서를 async 방식으로 로드
adocs = loader.aload()

In [40]:
# 문서 로드
await adocs

[Document(metadata={'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-09-17T06:14:24+00:00', 'title': '레퍼런스', 'moddate': '2026-09-17T06:14:24+00:00', 'source': './data/python_레퍼런스.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='📕\n레퍼런스\n외우지  않고  찾아보는  자리입니다 . 표 · 목록 · 문법  형식은  전부  여기로 .\n이해해야  하는  것은  개념 페이지 , 필요할  때  검색하는  것은  것은  레퍼런스\n🔗https://app.notion.com/p/3d747ecb9b23803abbb5e8abf2d722d6?\nsource=copy_link\n자주  쓰는  파이썬  함수\n텍스트  유형 : str[ 스트링 ]\n숫자  유형 : int[ 인티 ] , float , complex\n시퀀스  유형 : list[ 리스트 ] , tuple[ 튜플 ] , range[ 레인\n지 ]\n매핑  유형 : dict[ 딕트 ]\n세트  유형 : set ,frozenset\n부울  유형 : bool[ 부울 ] true/false ( 참 / 거짓 )\n이진  유형 : bytes , bytearray , memoryview\n없음  유형 : NoneType\nExample Data Type 중요도\nx = "Hello World" str O\nx = 20 int O\nx = 20.5 float O\nx = 1j complex\nx = ["apple", "banana", "cherry"] list O\nx = ("apple", "ba